# Assignment 1 — training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file — `main.py` refuses to run
if it finds both `train.py` and `train.ipynb`, so exactly one of them must
exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py train`, this notebook must, start to finish, with
no manual steps in between:

- Load your committed dataset.
- Apply the feature engineering, preprocessing, and train/val/test split
  consistent with the business framing you wrote in `REPORT.md`.
- Train the same model — linear regression for a regression target,
  logistic regression for a binary one — three ways, on the same split:
  scikit-learn, a from-scratch PyTorch loop (Session 5's manual approach:
  raw tensors, autograd, a manual gradient-step update), and the standard
  `torch.nn.Module` + `torch.optim` workflow.
- Evaluate all three against each other and against a naive baseline.
- Save whatever `app.py`/`app.ipynb` needs to build its dashboard without
  retraining anything.

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

## 1) Data preparation and feature engineering

In [13]:
#Inspect the dataset
import pandas as pd 
import numpy as np

df = pd.read_csv("data/hotel_bookings.csv")

df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [14]:
df.shape #(rows,columns)

(119390, 32)

In [15]:
df.columns.tolist() #list of columns

['hotel',
 'is_canceled',
 'lead_time',
 'arrival_date_year',
 'arrival_date_month',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'reserved_room_type',
 'assigned_room_type',
 'booking_changes',
 'deposit_type',
 'agent',
 'company',
 'days_in_waiting_list',
 'customer_type',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'reservation_status',
 'reservation_status_date']

In [16]:
df["hotel"].value_counts() #count per hotel (the dataset has 2 hotels, H1 and H2, which are a City Hotel and a Resort)

hotel
City Hotel      79330
Resort Hotel    40060
Name: count, dtype: int64

In [17]:
df["arrival_date_year"].value_counts().sort_index() #Data per year

arrival_date_year
2015    21996
2016    56707
2017    40687
Name: count, dtype: int64

In [18]:
#Filtering to only consider data from Resort Hotel & 2016 (aprox 18000 rows, so perfect for the 20k objective). We are therefore building a model for a specific hotel, so the business context is much more bounded.
df = df[
    (df["hotel"] == "Resort Hotel") &
    (df["arrival_date_year"] == 2016)
].copy()

df.shape

(18567, 32)

In [19]:
df["is_canceled"].value_counts(normalize=True) #proportion of cancelled vs not-cancelled. The class is much less imbalanced than I initially imagined.

is_canceled
0    0.734475
1    0.265525
Name: proportion, dtype: float64

In [20]:
df.isna().sum().sort_values(ascending=False) #check NAs per column

company                           17228
agent                              3921
country                             267
hotel                                 0
arrival_date_month                    0
arrival_date_week_number              0
lead_time                             0
is_canceled                           0
stays_in_weekend_nights               0
stays_in_week_nights                  0
children                              0
adults                                0
babies                                0
meal                                  0
arrival_date_day_of_month             0
arrival_date_year                     0
distribution_channel                  0
market_segment                        0
previous_bookings_not_canceled        0
is_repeated_guest                     0
reserved_room_type                    0
assigned_room_type                    0
booking_changes                       0
previous_cancellations                0
deposit_type                          0


In [21]:
df["country"] = df["country"].fillna("Unknown") #Instead of deleting the rows that hace "country" missing, we just put "unknown" there. Agent and company are floats (codes I guess) so we dont touch them.

In [22]:
df.info()

<class 'pandas.DataFrame'>
Index: 18567 entries, 3738 to 31985
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           18567 non-null  str    
 1   is_canceled                     18567 non-null  int64  
 2   lead_time                       18567 non-null  int64  
 3   arrival_date_year               18567 non-null  int64  
 4   arrival_date_month              18567 non-null  str    
 5   arrival_date_week_number        18567 non-null  int64  
 6   arrival_date_day_of_month       18567 non-null  int64  
 7   stays_in_weekend_nights         18567 non-null  int64  
 8   stays_in_week_nights            18567 non-null  int64  
 9   adults                          18567 non-null  int64  
 10  children                        18567 non-null  float64
 11  babies                          18567 non-null  int64  
 12  meal                            18567 non-nul

In [23]:
df["total_nights"] = ( #duración total de la estancia. inicialmente salía separada por weekend y week, también puede ser útil tener la estancia total, ya que el length of stay puede ser un buen predictor.
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

df["party_size"] = ( #total party size también puede ser un buen predictor.
    df["adults"] +
    df["children"].fillna(0) +
    df["babies"]
)

df["has_children"] = ( #boolean para separar entre "tiene niños" o "no"
    (df["children"].fillna(0) + df["babies"]) > 0
).astype(int)

df["has_previous_cancellation"] = (  #boolean para ver si previamente ha cancelado. Los valores originales obligan a la logistic regression a asumir algo parecido a que pasar de 0 a 1 cancelaciones tiene un efecto comparable a pasar de 2 a 3, cuando quizás aporte más info simplemente si ha cancelado alguna vez o no.
    df["previous_cancellations"] > 0
).astype(int)

df["has_special_requests"] = ( #boolean para ver si tiene special requests. Puede servir como proxy para ver si tienen "más compromiso" con la reserva
    df["total_of_special_requests"] > 0
).astype(int)

df["arrival_date"] = pd.to_datetime(  #Esto convierte año + mes + día en una fecha real de pandas, ya que necesitamos una fecha operable matemáticamente, no tres columnas separadas.
    df["arrival_date_year"].astype(str)
    + "-"
    + df["arrival_date_month"]
    + "-"
    + df["arrival_date_day_of_month"].astype(str)
)

df["booking_date"] = ( #El dataset no tiene directamente la fecha de reserva, pero sí arrival_date y lead_time. Reconstruyendo booking_date podemos ordenar cronológicamente las observaciones. Eso nos permitirá hacer un time-based train/validation/test split.
    df["arrival_date"] -
    pd.to_timedelta(df["lead_time"], unit="D")
)


I have decided that I want the model to make the predictions at booking time, so I have dropped certain columns to avoid data leakage (and also others that dont add much information).

In [24]:
columns_to_drop = [
    "hotel",                    # always Resort Hotel
    "arrival_date_year",        # always 2016
    "reservation_status",       # direct leakage
    "reservation_status_date",  # direct leakage
    "assigned_room_type",       # assigned later
    "booking_changes",          # can happen after booking
    "days_in_waiting_list",     # not reliably known at initial booking
    "company",                  # 93% missing + high-cardinality ID
    "agent"                     # ID rather than meaningful business feature
]

df = df.drop(columns=columns_to_drop)

## 2) Train, validation and test split

In [ ]:
# Sort bookings chronologically
df = df.sort_values("booking_date").reset_index(drop=True)

df[["booking_date", "is_canceled"]].head()

,booking_date,is_canceled
0,2014-03-18,0
1,2015-01-30,1
2,2015-01-30,1
3,2015-01-30,1
4,2015-01-30,1


In [ ]:
# Define a 70% / 15% / 15% chronological split. The temporal split is what better simulates real deployment.
#The data is split chronologically into 70% training, 15% validation and 15% test sets: training data is used to fit the model, validation data to make development decisions, and test data to evaluate final performance on unseen future bookings.
n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

In [35]:
# Create train, validation and test sets. iloc selecciona filas por posición. He hecho que no haya overlap entre los días de train, test y validation. (es decir, si )
train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

train_cutoff = train_df["booking_date"].max()
val_cutoff = val_df["booking_date"].max()

train_df = df[df["booking_date"] < train_cutoff].copy()

val_df = df[
    (df["booking_date"] >= train_cutoff) & #las fechas para el validation set tienen que ser mayor o igual al cut-off de train, y menor que el de test.
    (df["booking_date"] < val_cutoff)
].copy()

test_df = df[df["booking_date"] >= val_cutoff].copy()

In [36]:
#Just to check the sixzes of each split
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (12988, 30)
Validation: (2792, 30)
Test: (2787, 30)


In [37]:
print(
    "Train:",
    train_df["booking_date"].min(),
    "to",
    train_df["booking_date"].max()
)

print(
    "Validation:",
    val_df["booking_date"].min(),
    "to",
    val_df["booking_date"].max()
)

print(
    "Test:",
    test_df["booking_date"].min(),
    "to",
    test_df["booking_date"].max()
)

Train: 2014-03-18 00:00:00 to 2016-06-07 00:00:00
Validation: 2016-06-08 00:00:00 to 2016-09-17 00:00:00
Test: 2016-09-18 00:00:00 to 2016-12-31 00:00:00


## 3) Preprocessing

In [ ]:
# Variables used by the model. I have changed some variables for others (e.g. adults, children, babies for party_size and has_children). Si las metemos todas, habría redundacia entre ellas.

numeric_features = [
    "lead_time",
    "is_repeated_guest",
    "previous_bookings_not_canceled",
    "adr",
    "required_car_parking_spaces",
    "total_nights",
    "party_size",
    "has_children",
    "has_previous_cancellation",
    "has_special_requests"
]

categorical_features = [
    "arrival_date_month",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "deposit_type",
    "customer_type"
]

target = "is_canceled"

In [39]:
# Separate predictors (X) from target (y)
features = numeric_features + categorical_features

X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

X_test = test_df[features]
y_test = test_df[target]

In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:
# Define preprocessing for numerical and categorical variables
# I used one-hot encoding for categorical variables because logistic regression requires numerical inputs, and I did not want to impose an artificial ordinal relationship between categories.
# 

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(), #mean=0, std=1.
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore", #this is set to ignore so that unseen categories in validation or test would not cause the preprocessing pipeline to fail.
                sparse_output=False #we dont want compression, so that the conversion to a tensor if better afterwards.
            ),
            categorical_features
        )
    ]
)

In [ ]:
# Fit preprocessing ONLY on the training set
## Fit only on the training set so the preprocessing does not learn anything from validation or test data.
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
#check the shapes. 
#train, validation y test tienen el mismo número de columnas y el número de filas coincide con cada split, así que todo ok.
print("Train processed:", X_train_processed.shape)
print("Validation processed:", X_val_processed.shape)
print("Test processed:", X_test_processed.shape)

Train processed: (12988, 139)
Validation processed: (2792, 139)
Test processed: (2787, 139)


In [44]:
feature_names = preprocessor.get_feature_names_out()

print("Number of final features:", len(feature_names))
feature_names[:20]

Number of final features: 139


array(['num__lead_time', 'num__is_repeated_guest',
       'num__previous_bookings_not_canceled', 'num__adr',
       'num__required_car_parking_spaces', 'num__total_nights',
       'num__party_size', 'num__has_children',
       'num__has_previous_cancellation', 'num__has_special_requests',
       'cat__arrival_date_month_April', 'cat__arrival_date_month_August',
       'cat__arrival_date_month_December',
       'cat__arrival_date_month_February',
       'cat__arrival_date_month_January', 'cat__arrival_date_month_July',
       'cat__arrival_date_month_June', 'cat__arrival_date_month_March',
       'cat__arrival_date_month_May', 'cat__arrival_date_month_November'],
      dtype=object)

In [ ]:
#do we have missign values??? NO
print("NaNs in train:", np.isnan(X_train_processed).sum())
print("NaNs in validation:", np.isnan(X_val_processed).sum())
print("NaNs in test:", np.isnan(X_test_processed).sum())

NaNs in train: 0
NaNs in validation: 0
NaNs in test: 0
